# Paper Tables

This notebook is a thin wrapper around `mdu.eval.paper_tables`. Run `scripts/full_evaluation.py` first, then run the cells below to generate CSV and LaTeX tables for the paper.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mdu.eval.paper_tables import (
    PROBLEM_LABELS,
    build_and_write_paper_tables,
    build_llm_selective_generation_pareto_table,
)

INPUT_CSV = ROOT / "resources/refactored/results.csv"
LLM_RESULTS_DIR = ROOT / "resources/llm_resources"
OUTPUT_DIR = ROOT / "resources/paper_tables"
PARETO_BY_TASK_CSV = OUTPUT_DIR / "article_pareto_by_task_plot_data.csv"
PARETO_BY_TASK_FIG = OUTPUT_DIR / "article_pareto_by_task_barplot.pdf"

tables = build_and_write_paper_tables(INPUT_CSV, OUTPUT_DIR)
print(f"Wrote paper tables to {OUTPUT_DIR}")


In [ ]:
tables.average_ranks.head(20)

In [ ]:
for problem_type, table in tables.problem_mean_tables.items():
    print(problem_type, table.shape)
    display(table.round(3))

In [ ]:
tables.measure_summary.head(30)

In [ ]:
tables.article_pareto_latex_table

## Pareto Front by Task Type

This plot uses the same Pareto-front criterion as the Table 2-style table above, but splits the hit rate by task type. Image tasks use the article composition configured in `mdu.eval.paper_tables`; selective generation is computed from the LLM result CSVs when they are available.


In [ ]:
METHOD_ORDER = ["Ours", "Additive"]
TASK_ORDER = [
    "ood_detection",
    "misclassification_detection",
    "selective_prediction",
    "selective_generation",
]
TASK_LABELS = {
    **dict(PROBLEM_LABELS),
    "ood_detection": "OoD",
}
PALETTE = {
    "Ours": "#0072B2",
    "Additive": "#009E73",
}


In [ ]:
image_pareto_by_task = tables.article_pareto_by_problem_table.copy()
llm_pareto_by_task = build_llm_selective_generation_pareto_table(LLM_RESULTS_DIR)

pareto_by_task = pd.concat(
    [image_pareto_by_task, llm_pareto_by_task],
    ignore_index=True,
)
pareto_by_task = pareto_by_task[
    pareto_by_task["method"].isin(METHOD_ORDER)
    & pareto_by_task["problem_type"].isin(TASK_ORDER)
].copy()
pareto_by_task["problem_label"] = pareto_by_task["problem_type"].map(TASK_LABELS)
pareto_by_task["method"] = pd.Categorical(
    pareto_by_task["method"],
    categories=METHOD_ORDER,
    ordered=True,
)
pareto_by_task["problem_type"] = pd.Categorical(
    pareto_by_task["problem_type"],
    categories=TASK_ORDER,
    ordered=True,
)
pareto_by_task = pareto_by_task.sort_values(["problem_type", "method"])
pareto_by_task.to_csv(PARETO_BY_TASK_CSV, index=False)

display(
    pareto_by_task[
        [
            "problem_label",
            "method",
            "pareto_count",
            "total_pairs",
            "pareto_percentage",
            "average_pareto_depth",
        ]
    ]
)
print(f"Saved plot data to {PARETO_BY_TASK_CSV}")


In [ ]:
def plot_pareto_by_task_barplot(plot_df, output_path):
    if plot_df.empty:
        print("No Pareto-by-task data available; skipped plot.")
        return None

    present_tasks = [task for task in TASK_ORDER if task in set(plot_df["problem_type"].astype(str))]
    x = np.arange(len(present_tasks))
    width = 0.34

    with plt.rc_context({
        "text.usetex": False,
        "font.family": "serif",
        "font.serif": ["DejaVu Serif"],
        "font.size": 7,
        "axes.labelsize": 7,
        "axes.titlesize": 7.5,
        "legend.fontsize": 6.5,
        "xtick.labelsize": 6.5,
        "ytick.labelsize": 6.5,
        "axes.linewidth": 0.6,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }):
        fig, ax = plt.subplots(figsize=(5.1, 2.35), constrained_layout=True)
        for method_idx, method in enumerate(METHOD_ORDER):
            values = []
            counts = []
            for task in present_tasks:
                row = plot_df[
                    plot_df["problem_type"].astype(str).eq(task)
                    & plot_df["method"].astype(str).eq(method)
                ]
                if row.empty:
                    values.append(0.0)
                    counts.append(0)
                else:
                    values.append(float(row.iloc[0]["pareto_percentage"]))
                    counts.append(int(row.iloc[0]["total_pairs"]))

            offset = (method_idx - (len(METHOD_ORDER) - 1) / 2) * width
            bars = ax.bar(
                x + offset,
                values,
                width=width,
                label=method,
                color=PALETTE[method],
                edgecolor="white",
                linewidth=0.45,
                zorder=3,
            )
            labels = [f"{value:.0f}" if count else "" for value, count in zip(values, counts)]
            ax.bar_label(bars, labels=labels, padding=1.5, fontsize=5.8)

        ax.set_xticks(x)
        ax.set_xticklabels([TASK_LABELS[task] for task in present_tasks])
        ax.set_ylim(0, 108)
        ax.set_ylabel("At Pareto front (%)")
        ax.set_title("Pareto-front hit rate by task type")
        ax.legend(frameon=False, loc="upper center", ncols=len(METHOD_ORDER))
        ax.set_axisbelow(True)
        ax.yaxis.grid(True, color="0.88", linewidth=0.5)
        ax.xaxis.grid(False)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)
        for spine in ("left", "bottom"):
            ax.spines[spine].set_linewidth(0.6)
        ax.tick_params(axis="both", width=0.6, length=2.5)
        fig.savefig(output_path, bbox_inches="tight", pad_inches=0.02)

    plt.show()
    return fig

plot_pareto_by_task_barplot(pareto_by_task, PARETO_BY_TASK_FIG)
print(f"Saved plot to {PARETO_BY_TASK_FIG}")
